# Tier 2: Two Links

Read [README.md](README.md) first. Run this notebook from the tier folder.

In [25]:
import sys
sys.path.insert(0, "..")

from common import check, load_locked
from common import xor_bytes
import helper

locked = load_locked(2)
ciphertext = bytes.fromhex(locked["ciphertext_hex"])
print(locked["method"])
print(locked["public_parameters"])

XOR with the reconciled key of the clean link
{'links': ['link_A', 'link_B'], 'n_qubits': 1024}


## Build and run your circuit

Build the circuit below and run it on Aer. Recover the key.

In [26]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import random

# Your circuit goes here.
circuit = None

qlist = list()
blist = list()
# qlist is for the measurements of every qubit circuit, and blist is a record for the original (unmeasured) qbits on their random basis.


for i in range(1024):
# Because Alice's two links are 1,024 qubits long, our guess work also needs to be 1,024 qubits long. Every single circuit is made up of one qubit and one bit. These are 
# all stored in a list that allows them to be called back and read from later.
    qc = QuantumCircuit(1, 1)
    # Creates a temporary storage of the new ciruit for the i'nth iteration of this loop.
    if (random.randint(0,1))==1:
        # Randomly, every qubit is chosen to be either put into superposition, or not. There's a 50/50 chance of this happening. This is because of the random assignment used
        # by Alice in the creation of the two links, we recreate in our bases, which is the basis of BB84. The qc.h(0) command puts the only qubit in the temporary storage into 
        # superposition, before appending it to the list. Whereas if the randint rolls a 0, it is simply put into the list without being put into superposition.
        # In other terms: 1 puts it into the diagonal X-basis through Hadamard, while 0 keeps it in standard Z-basis.
        qc.h(0)
        blist.append(1)
    else:
        blist.append(0)
    qc.measure([0], [0])
    qlist.append(qc)
    # The temporary qubit storage is appended with a measurement operation, before storing that result in the qlist, to be executed later during simulation.



channel =  helper.Channel("link_A")
# This is where the link is chosen to be analysed. One link will be listened to by Eve, while the other won't be. There's no real reason as to which one is chosen, and our
# group simply chose A first, which was the correct answer, resulting in us not needing to check link b. Eve's eavesdropping will disturb the bits, resulting in a change between
# the results generated independantly by Bob and Alice. (This uses the no-cloning theorum, which is essential to quantum cryptography.)

mb = channel.receive_all(qlist)
# Makes another list where our generated bits are attached to alice's incoming hidden incoming staes,
# running the simulator, and returning the actual bits.
mf = channel.finish()
# This reveals the 25% that make up the public qubits, along with alice's original bases for every qubit. This is used later for comparison. Finally, it also gives the data for 
# reconsiliation, allowing us to finally reveal the key in combination with a dictionary of the private qubits (which is processed in the next loop.)

print(mb)
print(mf)
#Simply prints out the results for developer sanity checking. Which was needed a lot during this process. 

ab = mf['alice_bases']
# Extracts Alice's base qubit results into its own list.
cb = mf['check_bits']
# Extracts the public bits that are checked and compared against for the following loop.
kb = {}
# Creates the dictionary that every non-public matching bit is stored inside of.

for j in range(1024):
    if blist[j] == ab[j]:
        # Checks if our generated integer list of bases is the same as Alice's incoming integer list of bases. (The BB84 shifting phase where random incorrect results must be discarded.)
        if j in cb:
            # Checks if the number we're checking, exists in the dictionary of the check bits (public test batch used to hunt for Eve.) As Eve simply evesdropping will disturb and change about 25% of the matching bases.
            if mb[j] != cb[j]:
                print("No match")
                # A high amount of incorrect matches between our bits, and the bits from Alice, will indicate that this is the incorrect link (the one that's being listened
                # on), and that the other list is correct. As mentioned before, this will reveal if Eve is listening into this link or not.
        else:
            kb[j] = mb[j]
            #This stores the correct, matching bits inside of the dictionary created earlier.

print(kb)
# More developer sanity checking.

rr = mf['reconcile']
# This extracts the reconcile line from the finished channel from earlier.
bk = helper.reconcile(kb, rr)
# Using the extracted reconcile, this is compared against the dictionary created by the last loop to finally output the key which will be used for decryption.
print(bk)

[0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 

## Check

Set your key. This cell decrypts, hashes the result, and compares it with `hashes.txt`.

In [27]:
key = bk  # the bytes returned by helper.reconcile

plaintext = None
if key is not None:
    plaintext = xor_bytes(ciphertext, key).decode("utf-8", errors="replace")

check(2, plaintext)
print(plaintext)

PASS: tier 2 hash matches. Well done.
QC-02: bring 3 buckeyes, a blue pen
